# Simulation algorithms

The aim of this tutorial is to implement some geostatistical algorithms.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.spatial import distance_matrix

np.random.seed(12345)

## 1. Cholesky algorithm

1) Write a function which computes the value of the exponential covariance for a set of distances **h**, the **sill** and the **scale** parameter.

In [ ]:
def covariance_expo(h, sill=1, scale=1):
    return sill * np.exp(-h / scale)

2) Computation of the covariance matrix between points an a grid.

In [ ]:
ndiscr = 100
u = np.linspace(0, 100, ndiscr)
v = np.linspace(0, 100, ndiscr)

# Grid creation
coords = np.array([(i, j) for i in u for j in v])

# Distance matrix
dist = distance_matrix(coords, coords)

# Covariance matrix
covmat = covariance_expo(dist, scale=10)

3) Create and display a simulation of a GRF with mean 0 and the previous covariance matrix.

In [ ]:
simu = np.linalg.cholesky(covmat) @ np.random.normal(size=covmat.shape[1])
plt.imshow(simu.reshape((len(u), len(v))))
a = plt.colorbar()

4) Relaunch by increasing **ndiscr**. Be careful...

5) Simulate 1000 random vectors $(Z(x),Z(y))$ with $||x-y||=1$. Display the scatter plot and the regression line.

In [ ]:
coords = np.array(((0, 0), (0, 1)))
dist = distance_matrix(coords, coords)
covmat = covariance_expo(dist, scale=10)
result = np.linalg.cholesky(covmat) @ np.random.normal(
    size=covmat.shape[1] * 1000
).reshape((2, 1000))
plt.scatter(result[0, :], result[1, :], s=1)
rho = covmat[0, 1] / covmat[0, 0]
plt.plot([-3, 3], [-3 * rho, 3 * rho])

## 2. Spectral method

###  Construction of a simulation function

1) First, we create a function that will return **n** random objects **omega** and **phi** (as in the slides) in the Gaussian covariance case, from the **scale** parameter and the space **dimension**.


In [ ]:
def SpectrGaussObjects(n, scale, dimension=2):
    omega = np.random.normal(size=(n, dimension), loc=0, scale=np.sqrt(2) / scale)
    phi = 2.0 * np.pi * np.random.uniform(size=n)
    return omega, phi

Call

In [ ]:
omega, phi = SpectrGaussObjects(200, 6)

In [ ]:
ndiscr = 100
u = np.linspace(0, 100, ndiscr)
v = np.linspace(0, 100, ndiscr)

# Grid creation
coords = np.array([(i, j) for i in u for j in v])

2) Then, we create a function which will return the value of one band spread at (**coords**) locations from the random objects **phi** and **omega** (these objects can have been simulated with the previous function or with another one if you want to simulate another covariance function).

In [ ]:
def SpectrBasic(coords, phi, omega):
    return np.sqrt(2) * np.cos(coords @ omega.T + phi)

In [ ]:
SpectrBasic(coords, phi, omega).shape

3) Write the function which will return the final simulation at **coords** locations from the random objects **phi** and **omega** (as the previous one, these function can be used to simulate any covariance).

In [ ]:
def SpectrFinal(coords, phi, omega):
    mat = SpectrBasic(coords, phi, omega)
    return 1.0 / np.sqrt(mat.shape[1]) * np.sum(mat, axis=1)

Display

In [ ]:
plt.imshow(SpectrFinal(coords, phi, omega).reshape(len(u), len(v)))
plt.colorbar()

## With gstlearn (conditional simulations)

In [ ]:
import gstlearn.plot as gp
import minigst as mg

## Dataset
jura, grid, _ = mg.data("Jura")

First we need a model and a grid:

In [ ]:
## Create Db containing the training data
db_jura = mg.df_to_db(jura, ["Xloc", "Yloc"])
## Create experimental variogram
varioexp = mg.vario_exp(db_jura, vname="Co", nlag=30, dlag=0.1, toldis=0.5)

## Fit experimental variogram with an exponential covariance
types = ["EXPONENTIAL"]
fitmod = mg.model_fit(varioexp, struct=types)

## Create DbGrid that covers the Db containing the data
db_grid = mg.df_to_dbgrid(grid, ["Xloc", "Yloc"])

In [ ]:
mg.simulate_gauss_rf(db_grid, fitmod, nsim=1)
db_grid.display()

In [ ]:
## Plot Simulation
im = gp.plot(db_grid, "Simu")
gp.plot(db_jura, c="black", s=1)
plt.title("Unconditional Gaussian Random Field Simulation")
plt.axis("equal")
plt.colorbar(im)
plt.show()

For conditional simulations, we also need a conditioning dataset, here Jura and the cobalt variable "Co".

In [ ]:
mg.simulate_gauss_rf(db_grid, fitmod, nsim=1, dbcond=db_jura, vcond="Co")
db_grid.display()

In [ ]:
## Plot conditional Simulation
im = gp.plot(db_grid, "Simu.Co")
gp.plot(db_jura, c="black", s=1)
plt.title("Conditional Gaussian Random Field Simulation")
plt.axis("equal")
plt.colorbar(im)
plt.show()

In [ ]:
db_grid